# Investigation #3 — The SAE Replication Crisis Check

**Question:** When you train the same Top-K SAE with identical hyperparameters but
different random seeds, do you get the same feature dictionary?

The SAE literature implicitly assumes yes. Papers report feature counts, dead-neuron
rates, and qualitative feature descriptions as if they were properties of the model
rather than of a particular training run. If features are highly seed-dependent,
then many such claims are on shakier ground than advertised.

This notebook trains five SAEs on gpt2-small (layer 0, residual stream) with seeds
1–5, then computes pairwise bipartite matching across all 10 pairs using the
Hungarian algorithm on decoder-direction cosine similarity.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Point at the project root so imports resolve without installing
PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Path to the stability report produced by `mech analyze-sae-stability`
REPORT_PATH = PROJECT_ROOT / ".claude/worktrees/agent-a00cfb9423b24b13a/artifacts/seed_stability_report.json"

with REPORT_PATH.open() as fh:
    report = json.load(fh)

print(f"Loaded report: {report['summary']['n_pairs']} pairs, "
      f"{report['summary']['n_runs']} runs")

## 1. The Question and Why It Matters

Sparse autoencoders (Cunningham et al. 2023; Bricken et al. 2023; Gao et al. 2024)
decompose residual-stream activations into an overcomplete dictionary of directions,
each treated as a "feature". The standard workflow:

1. Train an SAE on activations from a frozen LLM.
2. Inspect high-activating prompts to label each learned direction.
3. Report findings as properties of the model.

Step 3 is only valid if step 1 is **reproducible**: the same model should produce
the same (or at least highly correlated) feature dictionary regardless of which
random seed was used. Nobody has systematically tested this.

### Metric

We measure **decoder direction cosine similarity** between matched features.
Two features with cosine > 0.9 are operationally "the same direction" — a probe
trained on one would transfer almost perfectly to the other.

We use the **Hungarian (optimal bipartite) matching** to find the best possible
one-to-one alignment between two dictionaries, then report the distribution of
matched cosines.

## 2. The Five Runs — Per-Run Summary Stats

In [ ]:
# The sweep produced runs 1-5 in the worktree artifact directory.
# Per-run metrics are in result.json files.

ARTIFACT_ROOT = PROJECT_ROOT / ".claude/worktrees/agent-a00cfb9423b24b13a/artifacts"

run_stats = []
for i in range(1, 6):
    run_dir = ARTIFACT_ROOT / f"run-{i:06d}"
    result_path = run_dir / "result.json"
    spec_path = run_dir / "spec.json"
    if result_path.exists():
        result = json.loads(result_path.read_text())
        spec = json.loads(spec_path.read_text()) if spec_path.exists() else {}
        seed = spec.get("parameters", {}).get("seed", i)
        m = result.get("metrics", {})
        run_stats.append({
            "run_id": i,
            "seed": seed,
            "live_features": m.get("live_features", 0),
            "dead_features": m.get("dead_features", 0),
            "final_loss": m.get("final_loss", 0.0),
            "explained_variance": m.get("explained_variance", 0.0),
        })

print(f"{'Run':>4} {'Seed':>5} {'Live':>6} {'Dead':>6} {'Loss':>10} {'Var%':>8}")
print("-" * 45)
for r in run_stats:
    print(f"{r['run_id']:>4} {r['seed']:>5} {r['live_features']:>6.0f} "
          f"{r['dead_features']:>6.0f} {r['final_loss']:>10.6f} "
          f"{r['explained_variance']*100:>7.1f}%")

## 3. Pairwise Alignment Matrix (Median Cosine Heatmap)

In [ ]:
N = 5
matrix = np.full((N, N), 1.0)  # diagonal = perfect self-alignment

run_names = [f"seed={i}" for i in range(1, N + 1)]

for p in report["pairwise"]:
    a_idx = int(p["run_a_name"].split("-")[-1]) - 1
    b_idx = int(p["run_b_name"].split("-")[-1]) - 1
    matrix[a_idx, b_idx] = p["median_cosine"]
    matrix[b_idx, a_idx] = p["median_cosine"]  # symmetric

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(matrix, vmin=0.0, vmax=1.0, cmap="RdYlGn")
plt.colorbar(im, ax=ax, label="Median matched cosine")

ax.set_xticks(range(N))
ax.set_yticks(range(N))
ax.set_xticklabels(run_names, rotation=45, ha="right")
ax.set_yticklabels(run_names)
ax.set_title("SAE Seed Stability — Median Cosine Similarity\n"
             "gpt2-small layer 0, 128 features, k=8")

for i in range(N):
    for j in range(N):
        ax.text(j, i, f"{matrix[i, j]:.3f}", ha="center", va="center",
                fontsize=9, color="black" if matrix[i, j] > 0.5 else "white")

plt.tight_layout()

# Save for the findings doc
img_path = PROJECT_ROOT / "docs/investigations/sae_replication_heatmap.png"
img_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(img_path, dpi=120, bbox_inches="tight")
print(f"Saved heatmap to {img_path}")
plt.show()

## 4. Distribution of Best-Match Cosines (Histogram)

In [ ]:
# Collect all matched cosines across all 10 pairs
all_cosines = []
for p in report["pairwise"]:
    all_cosines.extend(p["all_cosines"])

all_cosines_arr = np.array(all_cosines)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: full distribution
axes[0].hist(all_cosines_arr, bins=50, color="steelblue", edgecolor="white", linewidth=0.5)
axes[0].axvline(np.median(all_cosines_arr), color="red", linestyle="--",
                label=f"Median = {np.median(all_cosines_arr):.3f}")
axes[0].axvline(0.9, color="orange", linestyle=":", label="Threshold = 0.9")
axes[0].set_xlabel("Matched cosine similarity")
axes[0].set_ylabel("Count")
axes[0].set_title("All 10 pairs × 128 matched features\n(1280 total matches)")
axes[0].legend()

# Right: only top-1 match per pair
top1 = [max(p["all_cosines"]) for p in report["pairwise"]]
axes[1].bar(range(len(top1)), sorted(top1, reverse=True), color="steelblue")
axes[1].axhline(0.9, color="orange", linestyle=":", label="Threshold = 0.9")
axes[1].set_xlabel("Pair rank (sorted by best cosine)")
axes[1].set_ylabel("Best matched cosine")
axes[1].set_title("Best single feature match per pair")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Global median cosine: {np.median(all_cosines_arr):.4f}")
print(f"Fraction >= 0.9:      {(all_cosines_arr >= 0.9).mean():.2%}")
print(f"Fraction >= 0.5:      {(all_cosines_arr >= 0.5).mean():.2%}")
print(f"Fraction >= 0.3:      {(all_cosines_arr >= 0.3).mean():.2%}")

## 5. Concrete Examples

In [ ]:
# Find the best pair
best_pair = max(report["pairwise"], key=lambda p: max(p["all_cosines"]))
best_cosine = max(best_pair["all_cosines"])

# Find a borderline match (~0.5 cosine)
all_matches = []
for p in report["pairwise"]:
    for c in p["all_cosines"]:
        all_matches.append((abs(c - 0.5), c, p["run_a_name"], p["run_b_name"]))
all_matches.sort()
_, border_cos, border_a, border_b = all_matches[0]

# Find the worst individual match
worst_match = min(all_cosines)

print("Example A — clearly the SAME feature (cosine ~ 1.0 would be ideal)")
print(f"  Pair: {best_pair['run_a_name']} x {best_pair['run_b_name']}")
print(f"  Best cosine: {best_cosine:.4f}")
tm = best_pair["top_matches"][0]
print(f"  Feature a={tm['a_idx']}, b={tm['b_idx']}")
print()
print(f"Example B — borderline (cosine closest to 0.5)")
print(f"  Pair: {border_a} x {border_b}")
print(f"  Cosine: {border_cos:.4f}")
print()
print(f"Example C — clearly DIFFERENT feature")
print(f"  Worst individual matched cosine: {worst_match:.4f}")

## 6. Conclusion: Stability Fraction and Median Cosine

In [ ]:
all_cosines_arr = np.array(all_cosines)

global_median = float(np.median(all_cosines_arr))
stable_frac_09 = float((all_cosines_arr >= 0.9).mean())
stable_frac_07 = float((all_cosines_arr >= 0.7).mean())
stable_frac_05 = float((all_cosines_arr >= 0.5).mean())

s = report["summary"]

print("=" * 55)
print("HEADLINE NUMBERS")
print("=" * 55)
print(f"  Global median cosine (all 1280 matched pairs): {global_median:.4f}")
print(f"  Median-of-medians across 10 pairs:             {s['median_of_medians']:.4f}")
print(f"  Mean-of-means across 10 pairs:                 {s['mean_of_means']:.4f}")
print()
print("  Stability fraction (% features with cosine >= threshold)")
print(f"    >= 0.9 (SAE literature 'same feature'): {stable_frac_09:.2%}")
print(f"    >= 0.7:                                 {stable_frac_07:.2%}")
print(f"    >= 0.5:                                 {stable_frac_05:.2%}")
print()

if global_median < 0.5:
    verdict = "UNSTABLE: the median matched feature is closer to random than aligned."
elif global_median < 0.7:
    verdict = "WEAKLY STABLE: most features partially align but rarely lock to the same direction."
else:
    verdict = "STABLE: features broadly reproduce across seeds."

print("Verdict:", verdict)

## 7. Caveats and Extrapolation Risk

This experiment is deliberately narrow. Before generalising:

| Limitation | Why it matters |
|---|---|
| Only one model (gpt2-small) | Larger models with more redundancy might yield more stable features |
| Only one layer (layer 0 residual stream) | Later layers have more structured representations; stability may differ |
| Only one corpus (100-doc openwebtext sample) | A richer corpus changes what activates each feature |
| Only 128 features | The literature uses 4k–128k features; with more features, each covers a narrower concept and may be harder to reproduce |
| Only 5 seeds | Covers the variance space, but not exhaustively |
| Dead feature confound | ~66% of features are dead; dead features are trivially aligned at cosine≈0 with random directions, deflating the median |

The dead-feature confound is especially important: if ~66 of 128 features are dead
in every run, the bipartite matching will pair dead features from run A with dead
features from run B, producing low cosines for spurious reasons. A cleaner
experiment would restrict matching to live features only.

In [ ]:
# How many live features are there per run?
print("Live features per run (average ~43/128 = 34%):")
for r in run_stats:
    live = r['live_features']
    total = live + r['dead_features']
    print(f"  seed={r['seed']}: {live:.0f}/{total:.0f} live ({live/total:.0%})")